In [35]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from core.event_impact_analyzer import analyze_events_impact, EventImpactParams
from core.plot_utils import plot_price_real, plot_2d_events
from core.stock_data_provider import get_stock_data

# ── Параметры анализа ─────────────────────────────────────────────────────
TICKERS = ['LKOH', 'GAZP', 'SBER', 'NVTK']

PARAMS = EventImpactParams(
    baseline_start='2020-07-01',
    baseline_end='2021-12-31',
    window=20,
)

# загружаем данные первого тикера для Dash-визуализации событий
stock_data = get_stock_data(TICKERS[0])
stock_data.head()

,DATE,OPEN,HIGH,LOW,CLOSE,VOL
0,2002-01-08,403.15,426.48,403.15,420.00,1385900.0
1,2002-01-09,369.50,433.35,369.50,426.61,1476737.0
2,2002-01-10,424.10,431.30,420.01,426.00,1566808.0
3,2002-01-11,425.00,431.70,423.00,428.01,808508.0
4,2002-01-14,424.90,424.90,415.50,418.00,824284.0


In [36]:
import duckdb

# подключаемся (in-memory, без файла)
con = duckdb.connect()

# читаем два csv
con.execute("""
            CREATE TABLE event_tags AS
            SELECT *
            FROM read_csv_auto('../data/db/event_tags.csv');
            """)

con.execute("""
            CREATE TABLE events AS
            SELECT *
            FROM read_csv_auto('../data/db/events.csv');
            """)

# извлекаем только санкционные события
query = """
        SELECT e.*
        FROM events e
                 JOIN event_tags t ON e.id = t.event_id
        WHERE t.tag_code = 'SANCTIONS'
          and e.date_start > '2022-01-01'
        """

sanctions_df = con.execute(query).df()
sanctions_df

,id,date_start,date_end,event
0,017f1eba-7c00-4a09-b0ae-7fb048f2376a,2022-02-21,2022-02-23,Принятие первого пакета санкций против России.
1,017f2b9a-7200-4e6e-bd82-9da51f575bd3,2022-02-24,2022-02-25,Принятие второго пакета санкций против России.
2,017f5c86-fc00-4bda-b58f-dc6b1ea4dc81,2022-02-26,2022-03-14,Принятие третьего пакета санкций против России.
3,017fbe5f-7000-43f6-b297-6ac26c2b21f3,2022-03-15,2022-04-04,Принятие четвертого пакета санкций против России.
4,01808c5d-f000-4a32-b589-5e71ec54e94b,2022-04-05,2022-06-02,Принятие пятого пакета санкций против России.
5,01819302-7400-4b2a-966f-0b2829f35a0a,2022-06-03,2022-07-15,Принятие шестого пакета санкций против России.
6,0182df2c-7200-4454-9b9f-fc96608eb0ce,2022-07-21,2022-10-04,Принятие седьмого пакета санкций против России.
7,01845ed6-7800-48d7-bd6a-3dece3d0483b,2022-10-06,2022-12-15,Принятие 8 пакета санкций против России.
8,0185cc79-fc00-4dd1-9323-96318a82560e,2022-12-16,2023-02-24,Принят 9 пакет санкций против России.
9,018aad4f-f200-4824-8c3d-fdb306b3cfc2,2023-06-23,2023-12-17,Принятие 11 пакета санкций против России.


In [37]:
app = plot_2d_events(stock_data['DATE'], stock_data['CLOSE'], sanctions_df)
app.run()

In [38]:
# ── Нормализованный график цены с поправкой на инфляцию ──────────────────
NORM_TICKER = 'LKOH'  # поменяй на любой из TICKERS

stock_data_norm = get_stock_data(NORM_TICKER)

app2 = plot_price_real(
    stock_data_norm,
    normalize_date='2022-02-24',
    events_df=sanctions_df,
    title=f'Реальная цена {NORM_TICKER} (с поправкой на инфляцию), нормализована к 24.02.2022 = 100',
)
app2.run()

In [39]:
# ── Анализ влияния санкционных пакетов на все тикеры ─────────────────────
records = []
for ticker in TICKERS:
    for result, (_, row) in zip(analyze_events_impact(ticker, sanctions_df, PARAMS), sanctions_df.iterrows()):
        nan = float('nan')
        records.append({
            'Тикер': ticker,
            'Событие': result.event_text if result else row['event'],
            'Дата': result.event_date.date() if result else pd.Timestamp(row['date_start']).date(),
            'Обрезано': result.clipped if result else True,
            'Дней до': result.days_before if result else 0,
            'Дней после': result.days_after if result else 0,
            'Доходность точечная, %': result.point_return_pct if result else nan,
            'Доходность средняя, %': result.avg_return_pct if result else nan,
            'CAR, %': result.car_pct if result else nan,
            'Волатильность до, %': result.vol_before_pct if result else nan,
            'Волатильность после, %': result.vol_after_pct if result else nan,
            'Коэф. волатильности': result.vol_ratio if result else nan,
            'Объём до': int(result.volume_before) if result else 0,
            'Объём после': int(result.volume_after) if result else 0,
            'Коэф. объёма': result.volume_ratio if result else nan,
        })

results_df = pd.DataFrame(records)
results_df

,Тикер,Событие,Дата,Обрезано,Дней до,Дней после,"Доходность точечная, %","Доходность средняя, %","CAR, %","Волатильность до, %","Волатильность после, %",Коэф. волатильности,Объём до,Объём после,Коэф. объёма
0,LKOH,Принятие первого пакета санкций против России.,2022-02-21,True,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN
1,LKOH,Принятие второго пакета санкций против России.,2022-02-24,True,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN
2,LKOH,Принятие третьего пакета санкций против России.,2022-02-26,True,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN
3,LKOH,Принятие четвертого пакета санкций против России.,2022-03-15,True,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN
4,LKOH,Принятие пятого пакета санкций против России.,2022-04-05,True,6,19,-12.53,-13.56,-13.55,5.973,4.598,0.77,386273,588187,1.52
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,NVTK,Принятие 15 пакета санкций против России.,2024-12-16,False,20,20,7.41,11.47,22.67,2.254,3.556,1.58,2724814,3412395,1.25
68,NVTK,Принятие 16 пакета санкций против России.,2025-02-24,False,20,20,26.06,10.12,-5.94,3.089,2.165,0.70,5098002,3804648,0.75
69,NVTK,Принятие 17 Пакет Санкций против России.,2025-05-20,False,20,20,-11.56,-9.80,-9.11,3.254,3.052,0.94,3437978,2897498,0.84
70,NVTK,Принятие 18 Пакет санкций против России.,2025-07-18,False,20,20,22.66,5.64,14.41,2.100,2.205,1.05,2642606,4744350,1.80


● Интерпретация колонок таблицы
---
Метрики доходности

Доходность точечная, % - Цена в первый день окна "до" vs цена в последний день окна "после".
Нестабильна — зависит от двух конкретных дней. Используется только для наглядности.

Доходность средняя, %
Средняя цена за 20 дней после события vs средняя цена за 20 дней до.


Надёжнее точечной — сглаживает случайные дни. Основная метрика доходности.

- +5% → акция в среднем торговалась на 5% выше после события
- -15% → рынок в среднем упал на 15% в окне после события

---
CAR (Cumulative Abnormal Return), %

Сумма аномальных дневных доходностей в окне "после".
Аномальная = фактическая − ожидаемая (baseline 2020–2021).

Самая важная метрика. Отвечает на вопрос: акция упала из-за события, или она и так бы упала?
- CAR = -15% → акция потеряла 15% сверх того, что было бы без события
- CAR = +0% при доходность средняя = -5% → рынок падал и без события, само событие не добавило эффекта
- CAR = -5% при доходность средняя = -15% → из 15% падения только 5% объясняется аномалией, остальные 10% — норма рынка того периода

---
Волатильность

Волатильность до/после, % — среднеквадратичное отклонение дневных доходностей в окне (в %).

Коэф. волатильности = после / до.
- `> 1` → рынок нервничал после события (неопределённость выросла).
- `< 1` → рынок успокоился.
Пример 4.18 у LKOH на пакет 9 → колебания выросли в 4 раза — явный сигнал стресса

---
Объём

Объём до/после — средний дневной объём торгов в акциях.

Коэф. объёма = после / до
- `> 1` → объём вырос, рынок реагировал активно (инвесторы перекладывались)
- `< 1` → объём упал, реакции почти не было
Пример: 1.84 → объём вырос почти в 2 раза — событие привлекло внимание

---
Как читать строку в целом

Пример: SBER, начало войны
avg = -50.5%,  CAR = -16.7%,  vol_ratio = 0.79,  vol_ratio = 0.79
→ Акция упала на 50.5% в среднем за 20 дней после.
→ Из них ~17% — именно аномальная реакция на событие, остальные ~33% — рыночный контекст того периода (биржа была закрыта, торги заморожены).
→ Волатильность снизилась — парадоксально, но объясняется заморозкой торгов: дни без торгов дают нулевые доходности, сжимая std.


In [40]:
import uuid
from dash import Dash, html, dcc, Input, Output, State
from core.plot_utils import _find_free_port

METRICS = ['CAR, %', 'Доходность средняя, %', 'Доходность точечная, %', 'Коэф. волатильности', 'Коэф. объёма']

app = Dash(f'sanctions_{uuid.uuid4().hex[:8]}')
app.layout = html.Div([
    html.Div([
        dcc.Dropdown(id='dd-ticker', options=TICKERS, value=TICKERS[0], clearable=False, style={'width': '130px'}),
        dcc.Dropdown(id='dd-metric', options=METRICS, value='CAR, %', clearable=False, style={'width': '340px'}),
    ], style={'display': 'flex', 'gap': '12px', 'padding': '12px 16px'}),
    dcc.Graph(id='bar'),
])

@app.callback(Output('bar', 'figure'), Input('dd-ticker', 'value'), Input('dd-metric', 'value'))
def update(ticker, metric):
    df = results_df[results_df['Тикер'] == ticker].sort_values('Дата').reset_index(drop=True)
    x = [f'Пакет {i+1}' for i in df.index]
    ref = 1 if 'Коэф' in metric else 0
    colors = ['green' if v >= ref else 'red' for v in df[metric]]
    fig = go.Figure(go.Bar(
        x=x,
        y=df[metric],
        marker_color=colors,
        customdata=list(zip(df['Событие'], df['Дата'])),
        hovertemplate='<b>%{customdata[0]}</b><br>%{customdata[1]} — ' + metric + ': %{y:.2f}<extra></extra>',
    ))
    fig.add_hline(y=ref, line_dash='dash', line_color='gray', line_width=1)
    fig.update_layout(title=f'{ticker} — {metric}', template='plotly_white', height=480,
                      xaxis_title='Пакет санкций', yaxis_title=metric)
    return fig

app.run(port=_find_free_port(), jupyter_mode='inline')

In [41]:
# ── Heatmap: CAR по всем тикерам и пакетам ───────────────────────────────
pivot = results_df.pivot_table(index='Дата', columns='Тикер', values='CAR, %').sort_index()
y_labels = [f'Пакет {i + 1}  ({d})' for i, d in enumerate(pivot.index)]

fig = go.Figure(go.Heatmap(
    z=pivot.values,
    x=pivot.columns.tolist(),
    y=y_labels,
    colorscale='RdYlGn',
    zmid=0,
    text=[[f'{v:.1f}%' if pd.notna(v) else 'н/д' for v in row] for row in pivot.values],
    texttemplate='%{text}',
    hovertemplate='%{y}<br>%{x}: %{z:.2f}%<extra></extra>',
    colorbar=dict(title='CAR, %'),
))

fig.update_layout(
    title='CAR по пакетам санкций и тикерам, %',
    template='plotly_white',
    height=620,
    xaxis_title='Тикер',
    yaxis_autorange='reversed',
)

fig

In [42]:
# ── Event-window chart: путь цены ±N дней вокруг каждого события ─────────
def compute_windows(ticker, events_df, window=20):
    """Возвращает dict {df_index: (Series, label, date_str)} для событий с данными."""
    stock = get_stock_data(ticker)
    stock['DATE'] = pd.to_datetime(stock['DATE'])
    prices = stock.sort_values('DATE').set_index('DATE')['CLOSE']
    t_days = prices.index

    result = {}
    for idx, row in events_df.iterrows():
        edate = pd.Timestamp(row['date_start'])
        pos = t_days.searchsorted(edate)
        if pos >= len(t_days):
            continue
        ref = prices.iloc[pos]
        if ref == 0:
            continue
        chunk = prices.iloc[max(0, pos - window): pos + window + 1]
        rel = list(range(max(0, pos - window) - pos, pos + window + 1 - pos))
        s = pd.Series(chunk.values / ref * 100, index=rel).reindex(range(-window, window + 1))
        result[idx] = (s, row['event'], str(edate.date()))
    return result


_ew_cache = {t: compute_windows(t, sanctions_df, PARAMS.window) for t in TICKERS}

_ew_options = [
    {'label': f"{row['event']}  —  {pd.Timestamp(row['date_start']).date()}", 'value': idx}
    for idx, row in sanctions_df.iterrows()
]
_ew_all_values = [o['value'] for o in _ew_options]

app_ew = Dash(f'event_window_{uuid.uuid4().hex[:8]}')
app_ew.layout = html.Div([
    html.Div([
        dcc.Dropdown(
            id='ew-ticker', options=TICKERS, value=TICKERS[0],
            clearable=False, style={'width': '130px', 'flexShrink': '0'},
        ),
        dcc.Dropdown(
            id='ew-events',
            options=_ew_options,
            value=_ew_all_values,
            multi=True,
            placeholder='Выберите события...',
            style={'flex': '1', 'minWidth': '0'},
        ),
        dcc.Checklist(
            id='ew-show-avg',
            options=[{'label': ' Среднее', 'value': 'show'}],
            value=['show'],
            style={'flexShrink': '0', 'whiteSpace': 'nowrap'},
        ),
    ], style={'padding': '12px 16px', 'display': 'flex', 'gap': '16px', 'alignItems': 'center'}),
    dcc.Store(id='ew-base-figure'),
    dcc.Graph(id='ew-chart', clear_on_unhover=True),
])


@app_ew.callback(
    Output('ew-base-figure', 'data'),
    Output('ew-chart', 'figure'),
    Input('ew-ticker', 'value'),
    Input('ew-events', 'value'),
    Input('ew-show-avg', 'value'),
)
def update_ew(ticker, selected_indices, show_avg):
    windows_dict = _ew_cache[ticker]
    days = list(range(-PARAMS.window, PARAMS.window + 1))
    selected = set(selected_indices or [])

    fig = go.Figure()
    valid_windows = []

    for idx, (w, label, date) in windows_dict.items():
        if idx not in selected:
            continue
        if w.notna().sum() >= 4:
            valid_windows.append(w)
        fig.add_trace(go.Scatter(
            x=days, y=w.values, mode='lines',
            line=dict(color='rgba(180,180,180,0.45)', width=1),
            hovertemplate=f'<b>{label}</b><br>{date}<br>день %{{x}}: %{{y:.1f}}<extra></extra>',
            meta={'type': 'event'},
            showlegend=False,
        ))

    if valid_windows and 'show' in (show_avg or []):
        avg = pd.DataFrame(valid_windows).mean()
        fig.add_trace(go.Scatter(
            x=days, y=avg.values, mode='lines',
            line=dict(color='steelblue', width=3),
            name='Среднее',
            hovertemplate='Среднее<br>день %{x}: %{y:.1f}<extra></extra>',
            meta={'type': 'avg'},
        ))

    fig.add_vline(x=0, line_dash='dash', line_color='crimson', line_width=1.5, annotation_text='событие')
    fig.add_hline(y=100, line_dash='dot', line_color='gray', line_width=1)
    fig.update_layout(
        title=f'{ticker} — путь цены в окне ±{PARAMS.window} дн. вокруг события (день 0 = 100)',
        xaxis_title='Дней до/после события',
        yaxis_title='Нормализованная цена (день 0 = 100)',
        template='plotly_white',
        height=500,
    )
    fig_dict = fig.to_dict()
    return fig_dict, fig_dict


app_ew.clientside_callback(
    """
    function(hoverData, baseFigure) {
        if (!baseFigure) return window.dash_clientside.no_update;
        const fig = JSON.parse(JSON.stringify(baseFigure));
        if (!hoverData || !hoverData.points || !hoverData.points.length) {
            return fig;
        }
        const curveNum = hoverData.points[0].curveNumber;
        fig.data.forEach(function(trace, i) {
            if (!trace.meta || trace.meta.type !== 'event') return;
            if (i === curveNum) {
                trace.line.color = 'rgba(70,130,180,0.95)';
                trace.line.width = 2.5;
            } else {
                trace.line.color = 'rgba(200,200,200,0.12)';
                trace.line.width = 1;
            }
        });
        return fig;
    }
    """,
    Output('ew-chart', 'figure', allow_duplicate=True),
    Input('ew-chart', 'hoverData'),
    State('ew-base-figure', 'data'),
    prevent_initial_call=True,
)

app_ew.run(port=_find_free_port(), jupyter_mode='inline')

In [43]:
# ── Сохранение результатов ────────────────────────────────────────────────
results_df.to_csv('../reports/sanctions_impact.csv', index=False)
print(f'Сохранено: {len(results_df)} строк → reports/sanctions_impact.csv')

Сохранено: 72 строк → reports/sanctions_impact.csv
